In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import string

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,SimpleRNN

import warnings
warnings.filterwarnings('ignore')

In [4]:
df=pd.read_csv("qoute_dataset.csv")
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [5]:
df.shape

(3038, 2)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3038 entries, 0 to 3037
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   quote   3038 non-null   object
 1   Author  3038 non-null   object
dtypes: object(2)
memory usage: 47.6+ KB


In [7]:
quotes= df['quote']
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


In [8]:
quotes= quotes.str.lower()

In [9]:
translator= str.maketrans('','',string.punctuation)
quotes= quotes.apply(lambda x:x.translate(translator))

In [10]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


In [11]:
vocal_size=8978

tok=Tokenizer(num_words=vocal_size)
tok.fit_on_texts(quotes)


In [12]:
word_index= tok.word_index
print(len(word_index))
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [13]:
sequence= tok.texts_to_sequences(quotes)

In [14]:
quotes[0]

'“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”'

In [15]:
sequence[0]

[713,
 62,
 29,
 19,
 16,
 946,
 10,
 7,
 5,
 1156,
 8,
 70,
 293,
 10,
 145,
 12,
 809,
 104,
 752,
 70,
 2461]

In [16]:
X= []
y= []

for seq in sequence:
    for i in range(1,len(seq)):
      input_seq= seq[:i]
      output_seq= seq[i]
      X.append(input_seq)
      y.append(output_seq)

In [17]:
len(X)

85270

In [18]:
len(y)

85270

In [19]:
max_len= max(len(x) for x in X)
max_len

745

In [20]:
X_padded= pad_sequences(X,maxlen=max_len,padding='pre')

In [21]:
y=np.array(y)

In [22]:
X_padded.shape

(85270, 745)

In [23]:
y.shape

(85270,)

In [24]:
y

array([ 62,  29,  19, ...,   3, 169, 101])

In [25]:
y_onehot= to_categorical(y,num_classes=vocal_size)
y_onehot

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [26]:
embedding= 50
rnn_units= 128
rnn_model= Sequential()
rnn_model.add(Embedding(input_dim=vocal_size,output_dim=embedding,input_length=max_len))
rnn_model.add(SimpleRNN(units=rnn_units,return_sequences=True))
rnn_model.add(Dense(units=vocal_size,activation='softmax'))

In [27]:
rnn_model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

In [28]:
rnn_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [29]:
lstm_model= Sequential()
lstm_model.add(Embedding(input_dim=vocal_size,output_dim=embedding,input_length=max_len))

lstm_model.add(LSTM(units=rnn_units,return_sequences=True))
lstm_model.add(Dense(units=vocal_size,activation='softmax'))

In [30]:
lstm_model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

In [31]:
lstm_model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [32]:
embedding= 50
rnn_units= 128
rnn_model= Sequential()
rnn_model.add(Embedding(input_dim=vocal_size,output_dim=embedding,input_length=max_len))
rnn_model.add(SimpleRNN(units=rnn_units,return_sequences=False)) # FIX: Changed to False
rnn_model.add(Dense(units=vocal_size,activation='softmax'))

rnn_model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

epochs=5
batch_size= 128
history_rnn=rnn_model.fit(X_padded,y_onehot,epochs=epochs,batch_size=batch_size,validation_split=0.1)

Epoch 1/5
600/600 ━━━━━━━━━━━━━━━━━━━━ 46s 70ms/step - accuracy: 0.0294 - loss: 7.5350 - val_accuracy: 0.0318 - val_loss: 7.4876
Epoch 2/5
600/600 ━━━━━━━━━━━━━━━━━━━━ 39s 64ms/step - accuracy: 0.0454 - loss: 6.7158 - val_accuracy: 0.0509 - val_loss: 6.7830
Epoch 3/5
600/600 ━━━━━━━━━━━━━━━━━━━━ 39s 64ms/step - accuracy: 0.0544 - loss: 6.3884 - val_accuracy: 0.0521 - val_loss: 6.7125
Epoch 4/5
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 64ms/step - accuracy: 0.0667 - loss: 6.1935 - val_accuracy: 0.0653 - val_loss: 6.6329
Epoch 5/5
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 64ms/step - accuracy: 0.0803 - loss: 5.9968 - val_accuracy: 0.0793 - val_loss: 6.5616


In [41]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# Redefine necessary variables (assuming their values are already established in the notebook flow)
vocal_size = 8978 # Value from cell tywd0_bC1N5A
embedding = 50   # Value from cell VMb1WBSe79V9
rnn_units = 128  # Value from cell VMb1WBSe79V9
max_len = 745    # Value from cell isQ9BxGl5sgX

# Redefine the LSTM model with return_sequences=False
lstm_model= Sequential()
lstm_model.add(Embedding(input_dim=vocal_size,output_dim=embedding,input_length=max_len))
lstm_model.add(LSTM(units=rnn_units,return_sequences=False)) # FIX: Changed to False
lstm_model.add(Dense(units=vocal_size,activation='softmax'))

# Recompile the model
lstm_model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

epochs=5
batch_size= 128
history_lstm=lstm_model.fit(X_padded,y_onehot,epochs=epochs,batch_size=batch_size,validation_split=0.1)

Epoch 1/5
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 53ms/step - accuracy: 0.0390 - loss: 6.7429 - val_accuracy: 0.0480 - val_loss: 6.6877
Epoch 2/5
600/600 ━━━━━━━━━━━━━━━━━━━━ 31s 51ms/step - accuracy: 0.0593 - loss: 6.3126 - val_accuracy: 0.0663 - val_loss: 6.5562
Epoch 3/5
600/600 ━━━━━━━━━━━━━━━━━━━━ 31s 52ms/step - accuracy: 0.0828 - loss: 6.0449 - val_accuracy: 0.0856 - val_loss: 6.4511
Epoch 4/5
600/600 ━━━━━━━━━━━━━━━━━━━━ 31s 52ms/step - accuracy: 0.0995 - loss: 5.8221 - val_accuracy: 0.0958 - val_loss: 6.4273
Epoch 5/5
600/600 ━━━━━━━━━━━━━━━━━━━━ 31s 52ms/step - accuracy: 0.1104 - loss: 5.6347 - val_accuracy: 0.1000 - val_loss: 6.4028


In [34]:
lstm_model.save('lstm_model.h5')
rnn_model.save('rnn_model.h5')

In [36]:
index_to_word= {}
for word,index in word_index.items():
  index_to_word[index]= word

In [44]:
def perdictor(model,tok,text,max_len):
  text= text.lower()
  seq= tok.texts_to_sequences([text])[0]
  seq= pad_sequences([seq],maxlen=max_len,padding='pre')

  pred= model.predict(seq,verbose=0)
  pred_index= np.argmax(pred)
  return index_to_word[pred_index]


In [50]:
seed_text="what is"
next_word= perdictor(lstm_model,tok,seed_text,max_len)
print(next_word)

a


In [57]:
def generate_text(model,tok,seed_text,n_words,max_len):
  for _ in range(n_words):
    next_word= perdictor(model,tok,seed_text,max_len)
    if next_word is "":
      break
    seed_text += " "+ next_word
  return seed_text

In [58]:
seed= "the meaning of life"
generate_text= generate_text(lstm_model,tok,seed,10,max_len)
print(generate_text)

the meaning of life is a book is a book is a world is
